[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/pascal/pascal.ipynb)

# Pascal's Theorem

This example finds the quadric through five points and checks Pascal's theorem on it. Five points in space, no three on a line, fix one quadric through them. Add a sixth point on that quadric and join the six in order into a hexagon: the three pairs of opposite sides meet in three points, and those three always lie on one line, however the hexagon is shaped, convex or crossed. Read backwards, the theorem builds the quadric: the sixth points for which the three crossings line up are exactly the points of the quadric through the other five.

A quadric is a map from points to planes, `Plane <- Point`, sending each point to its polar plane, and the quadric is the set of points that lie on their own polar, `quadric(point) & point == 0`. Two lines make a quadric, their pair, and the quadrics through four points are the combinations of two such pairs; a fifth point fixes the combination. Pascal's condition, three crossings of joins joined into a number, is a form on the sixth point, which appears in two of the hexagon's sides, and it is the quadric itself, up to scale.

In matrix notation a quadric reads as a symmetric $3\times3$ matrix $C$ with $\mathbf p^{\mathsf T} C\,\mathbf p = 0$ on its points, the pencil through four points as the combination of two rank-two matrices, and the three crossings on one line as a vanishing $3\times3$ determinant.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from numga.algebras import PGA2D as ga
from numga import NumpyContext
from examples.util.animation import save_figure, show_animation
from examples.math.pascal import render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
# Planes are vectors and points antivectors; a quadric sends a point to its polar plane.
Plane = ga.gatype.vector()                                                     # Vector+
Point = ga.gatype.antivector()                                                 # Antivector-
# The join of two points is a line, an intrinsic antibivector, and two such lines cross in an extrinsic bivector.
Line = ga.gatype.antibivector()                                                # Antibivector-
Crossing = ga.gatype.bivector()                                                # Bivector+
Quadric = ga.gatype((Plane, Point))                                            # Plane+ <- Point-
Form = ga.gatype((Scalar, Point, Point))                                       # Scalar+ <- (Point-, Point-)

## 1. The quadric through five points

Two lines make a quadric, the pair of them, whose value at a point is the product of the two lines' pairings with it, zero on either line. Through four points pass two pairs of lines, one pair of opposite sides of their quadrilateral and the pair of its diagonals, and every quadric through the four is a combination of them. The fifth point fixes the combination, the one that vanishes there.

In [ ]:
# The ellipsoid the five points are taken from: its semi-axes along x and y, its centre, and their angles on it.
semi_axes = np.array([1.6, 1.0])
centre = np.array([0.3, -0.1])
five_angles = np.array([0.4, 1.44, 4.12, 4.77, 6.08])

In [ ]:
def quadric(points: Point) -> Quadric:
    """The quadric through five points on the last axis: the combination of the two line pairs through
    the first four that vanishes on the fifth."""
    first, second, third, fourth, fifth = (points[..., index] for index in range(5))
    # Through the first four points pass two pairs of lines: two opposite sides of their quadrilateral,
    # and its two diagonals.
    lines = stack([first & second, first & third])                             # [pairs, ...] Line-
    partners = stack([third & fourth, second & fourth])                        # [pairs, ...] Line-
    # Each pair of lines as one quadric, its value at a point the product of the two lines' pairings with it.
    pairs = (lines * (partners & Point) + partners * (lines & Point)) / 2      # [pairs, ...] Plane+ <- Point-
    sides, diagonals = pairs[0], pairs[1]                                      # [...] Plane+ <- Point-, Plane+ <- Point-
    return sides * (diagonals(fifth) & fifth) - diagonals * (sides(fifth) & fifth)   # [...] Plane+ <- Point-


five = mv.point(centre + semi_axes * np.stack([np.cos(five_angles), np.sin(five_angles)], axis=-1))   # [5] Point-
shape = quadric(five)                                                          # [] Plane+ <- Point-

In [ ]:
print(shape.coefficient_table())

## 2. A sixth point, and the hexagon

A line through the first point meets the quadric once more. Along the line the quadric's value is a quadratic in the distance travelled with one root at the start, so the other root follows without a square root. Turning the line half a turn about the first point runs the sixth point once around the quadric. With it, the six points make a hexagon, and its three pairs of opposite sides meet in three points.

In [ ]:
# The turning line's angles, half a turn in steps, the step shown as a still, and the duration of a frame in
# milliseconds.
n_frames = 120
angles = np.linspace(0.0, np.pi, n_frames, endpoint=False) + np.pi / (2 * n_frames)
still = 72
frame_duration = 60
# The next corner of each corner of a hexagon.
next_corner = np.array([1, 2, 3, 4, 5, 0])

In [ ]:
start = five[0]                                                                # [] Point-
# Points at infinity in the directions of the lines through the start.
turns = (mv.xy * (-angles / 2)).exp()                                          # [frames] Motor+
headings = turns >> mv.x.dual()                                                # [frames] Point-
# Along the line, `start + heading * t`, the quadric's value is
# `2 * t * (shape(start) & heading) + t * t * (shape(heading) & heading)`: zero at the start, and at the other root.
sixth = start * (shape(headings) & headings) - headings * (2 * (shape(start) & headings))   # [frames] Point-
sixth = sixth.normalized()                                                     # [frames] Point-
hexagon = stack([five[index].broadcast_to(sixth.shape) for index in range(5)] + [sixth], axis=-1)   # [frames, 6] Point-
# Each side the join of a corner and the next; opposite sides meet in the three crossings, and the line through
# them is Pascal's.
sides = hexagon & hexagon[..., next_corner]                                    # [frames, 6] Line-
crossing = sides[..., :3] ^ sides[..., 3:]                                     # [frames, 3] Crossing+
pascal_line = crossing[..., 0] & crossing[..., 2]                              # [frames] Line-
save_figure(render.draw(shape, hexagon[still], sides[still], crossing[still], pascal_line[still]), "pascal_hexagon");

## 3. Three crossings on one line

As the sixth point runs around, the hexagon changes shape, and the three crossings move, some far out when opposite sides are nearly parallel; the line through them, red, turns with them and never loses one. Their join, the regressive product of all three, is zero at every step.

In [ ]:
show_animation(render.animate([(shape, hexagon[frame], sides[frame], crossing[frame], pascal_line[frame])
                               for frame in range(n_frames)]),
               "pascal_turn", frame_duration)

## 4. Pascal's condition as the quadric

Written as a condition on the sixth point, the three crossings joined into one number is a form on that point, which appears in two of the hexagon's sides, and so quadratic in it. It vanishes on every sixth point of the quadric, and on nothing else: at any point at all it is the quadric's value there, times one constant.

In [ ]:
# Points anywhere, on the quadric or off it, and their seed.
n_probes = 8
seed = 0
probes = mv.point(np.random.default_rng(seed).normal(size=(n_probes, 2)))        # [probes] Point-

In [ ]:
def pascal(five: Point) -> Form:
    """Pascal's condition on a sixth point after five, a form on the sixth point, which appears in two of
    the hexagon's sides: the join of the three crossings of opposite sides, zero exactly when they lie
    on one line."""
    first, second, third, fourth, fifth = (five[..., index] for index in range(5))
    return (((first & second) ^ (fourth & fifth))
            & ((second & third) ^ (fifth & Point))
            & ((third & fourth) ^ (Point & first)))                            # [...] Scalar+ <- (Point-, Point-)


ratio = pascal(five)(probes, probes) / (shape(probes) & probes)                # [probes] Scalar+

In [ ]:
print("Pascal's condition over the quadric, at each probe:", ratio.to_array())

In [ ]:
# checks
# The quadric passes through the five points and every sixth; the three crossings lie on one line, their join
# measured against their sizes, since nearly parallel sides meet far out; Pascal's condition is the quadric, up to
# one constant.
join = crossing[..., 0] & crossing[..., 1] & crossing[..., 2]                   # [frames] Scalar+
size = crossing[..., 0].norm() * crossing[..., 1].norm() * crossing[..., 2].norm()   # [frames] Scalar+
np.testing.assert_allclose((shape(five) & five).to_array(), 0.0, atol=1e-11)
np.testing.assert_allclose((shape(hexagon) & hexagon).to_array(), 0.0, atol=1e-11)
np.testing.assert_allclose((join / size).to_array(), 0.0, atol=1e-10)
np.testing.assert_allclose(ratio.to_array(), ratio.to_array()[0], rtol=1e-11)